In [ ]:
%matplotlib inline

# attempt_21_resnext_only.ipynb -- Diabetic Retinopathy Detection

Inference-only notebook: loads the best ResNeXt50-32x4d Stage 2 weights
(`best_resnext_s2.pth`, val AUC 0.8066) and generates the Codabench
submission file for the fine-tuning category.

## 1. Imports & Setup

In [ ]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnext50_32x4d, ResNeXt50_32X4D_Weights
import torch.nn as nn
from PIL import Image
from zipfile import ZipFile
import cv2
import warnings

warnings.filterwarnings('ignore')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_ROOT  = '/kaggle/input/datasets/mariamuozperez/lab5-cv'
MODEL_PATH = '/kaggle/working/best_resnext_s2.pth'  # adjust if uploaded as a dataset

## 2. Dataset & Transforms

In [ ]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

In [ ]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        img_u8 = image if image.dtype == np.uint8 else (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}

## 3. DataLoaders

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

val_dataloader  = DataLoader(val_dataset,  batch_size=256, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

print(f'Val: {len(val_dataset)}  Test: {len(test_dataset)}')

## 4. Load ResNeXt50 Model

In [ ]:
# Reconstruct the same architecture used during training
model = resnext50_32x4d(weights=None)
in_feats = model.fc.in_features  # 2048
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_feats, 1)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print(f'Loaded: {MODEL_PATH}')
total = sum(p.numel() for p in model.parameters())
print(f'ResNeXt50 total params: {total:,}')

## 5. Validation AUC (sanity check)

In [ ]:
scores_val = np.zeros((len(val_dataset), 1), dtype=np.float32)
labels_val = np.zeros((len(val_dataset),),   dtype=np.int64)
cont = 0
with torch.no_grad():
    for sample in val_dataloader:
        inputs = sample['image'].to(device).float()
        bs = inputs.shape[0]
        s1 = torch.sigmoid(model(inputs))
        s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
        s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
        s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
        out = (s1 + s2 + s3 + s4) / 4.0
        scores_val[cont:cont+bs, :] = out.cpu().numpy()
        labels_val[cont:cont+bs]    = sample['label'].numpy()
        cont += bs

val_auc = metrics.roc_auc_score(labels_val, scores_val)
print(f'Val AUC (TTA-4): {val_auc:.4f}  — expected ~0.8066')

## 6. Test Inference & Submit

In [ ]:
scores_test = np.zeros((len(test_dataset), 1), dtype=np.float32)
cont = 0
with torch.no_grad():
    for sample in test_dataloader:
        inputs = sample['image'].to(device).float()
        bs = inputs.shape[0]
        s1 = torch.sigmoid(model(inputs))
        s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
        s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
        s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
        out = (s1 + s2 + s3 + s4) / 4.0
        scores_test[cont:cont+bs, :] = out.cpu().numpy()
        cont += bs

assert scores_test.shape == (1000, 1)
assert np.isfinite(scores_test).all()
print(f'Test scores: shape={scores_test.shape}, min={scores_test.min():.4f}, max={scores_test.max():.4f}')

In [ ]:
with open('output_ft.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(scores_test)

with ZipFile('./codabench_submission.zip', 'w') as zf:
    zf.write('./output_ft.csv')

print('Created: codabench_submission.zip  (output_ft.csv only)')
print(f'Val AUC: {val_auc:.4f}')